In [1]:
import torch
import torch.nn as nn

## Summary Equation

$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \gamma + \beta$$

* **$\mu, \sigma^2$**: Computed per-sample across feature dimensions.
* **$\gamma, \beta \in \mathbb{R}^d$**: Learnable parameters initialized to $1$ and $0$.
* **$\epsilon$**: Small constant for numerical stability (typically $10^{-5}$).

In [2]:
class LayerNorm(nn.Module):
  def __init__(self, d_model, epsilon=1e-5):
    super(LayerNorm, self).__init__()

    self.epsilon = epsilon
    self.gamma = nn.Parameter(torch.ones(d_model))
    self.beta = nn.Parameter(torch.zeros(d_model))

  def forward(self, x: torch.Tensor):
    mean = x.mean(-1, keepdim=True)
    variance = x.var(-1, keepdim=True)

    x_hat = (x - mean) / torch.sqrt(variance + self.epsilon)
    y_hat = self.gamma * x_hat + self.beta

    return y_hat

  def backward(self, grad_output, x):

    x_hat = self.forward(x)
    variance = x.var(-1, keepdim=True)

    dbeta = grad_output.sum(dim=(0,1))
    dgamma = (grad_output * x_hat).sum(dim=(0,1))

    g = grad_output * self.gamma
    dx = (g - g.mean(dim=-1, keepdim=True) - x_hat * (g * x_hat).mean(dim=-1, keepdim=True)) / torch.sqrt(variance + self.epsilon)

    return dx, dgamma, dbeta


# Backpropagation Derivation for Layer Normalization

This document derives the complete backward pass for Layer Normalization from first principles.

---

## 1. Forward Pass

Consider an input vector:

$$x = (x_1, x_2, \dots, x_D)$$

where LayerNorm normalizes across the feature dimension $D$.

### Mean
$$\mu = \frac{1}{D}\sum_{i=1}^{D}x_i$$

### Variance
$$\sigma^2 = \frac{1}{D}\sum_{i=1}^{D}(x_i-\mu)^2$$

### Standard Deviation
$$s = \sqrt{\sigma^2+\epsilon}$$

### Normalization
$$\hat{x}_i = \frac{x_i-\mu}{s}$$

### Affine Transformation
$$y_i = \gamma_i\hat{x}_i+\beta_i$$

Assume the upstream gradient is:

$$g_i = \frac{\partial L}{\partial y_i}$$

Our objective is to compute $\frac{\partial L}{\partial x_i}$.

---

## 2. Gradients with respect to $\beta$ and $\gamma$

Since $y_i = \gamma_i\hat{x}_i+\beta_i$, the gradients are immediate:

* **Gradient of $\beta$:**
  $$\frac{\partial L}{\partial\beta_i} = g_i$$

* **Gradient of $\gamma$:**
  $$\frac{\partial L}{\partial\gamma_i} = g_i\hat{x}_i$$

---

## 3. Gradient with respect to normalized input

Using the chain rule:

$$\frac{\partial L}{\partial\hat{x}_i} = g_i\gamma_i$$

For convenience, define:

$$d_i = \frac{\partial L}{\partial\hat{x}_i} = g_i\gamma_i$$

---

## 4. Derivative of the Mean

The mean is $\mu = \frac1D\sum_{k=1}^{D}x_k$. Differentiating with respect to any input $x_j$:

$$\frac{\partial\mu}{\partial x_j} = \frac1D \sum_k \frac{\partial x_k}{\partial x_j}$$

Since $\frac{\partial x_k}{\partial x_j} = \delta_{kj}$ (where $\delta_{kj} = 1$ if $k=j$ and $0$ otherwise), we obtain:

$$\frac{\partial\mu}{\partial x_j} = \frac1D$$

---

## 5. Derivative of the Standard Deviation

Recall $s = (\sigma^2+\epsilon)^{1/2}$. Applying the chain rule to find $\frac{\partial s}{\partial x_j}$:

$$\frac{\partial s}{\partial x_j} = \frac{\partial s}{\partial\sigma^2} \cdot \frac{\partial\sigma^2}{\partial x_j}$$

Since $\frac{\partial s}{\partial\sigma^2} = \frac12 (\sigma^2+\epsilon)^{-1/2} = \frac1{2s}$, we get:

$$\frac{\partial s}{\partial x_j} = \frac1{2s} \frac{\partial\sigma^2}{\partial x_j}$$

---

## 6. Derivative of $1/s$

Since $\hat{x}_i = (x_i-\mu)s^{-1}$, we require $\frac{\partial s^{-1}}{\partial x_j}$:

$$\frac{\partial s^{-1}}{\partial x_j} = \frac{\partial s^{-1}}{\partial s} \frac{\partial s}{\partial x_j} = -\frac1{s^2} \left( \frac1{2s} \frac{\partial\sigma^2}{\partial x_j} \right)$$

Hence:

$$\frac{\partial s^{-1}}{\partial x_j} = -\frac1{2s^3} \frac{\partial\sigma^2}{\partial x_j}$$

---

## 7. Derivative of the Variance

Variance is $\sigma^2 = \frac1D \sum_k (x_k-\mu)^2$. Differentiating with respect to $x_j$:

$$\frac{\partial\sigma^2}{\partial x_j} = \frac1D \sum_k 2(x_k-\mu) \frac{\partial(x_k-\mu)}{\partial x_j} = \frac2D \sum_k (x_k-\mu) \left( \delta_{kj} - \frac1D \right)$$

Expanding the sum:

$$\frac{\partial\sigma^2}{\partial x_j} = \frac2D \left[ \sum_k(x_k-\mu)\delta_{kj} - \frac1D \sum_k(x_k-\mu) \right]$$

The second term vanishes because $\sum_k(x_k-\mu) = 0$. The first term selects the $j$-th element:

$$\frac{\partial\sigma^2}{\partial x_j} = \frac2D(x_j-\mu)$$

---

## 8. Jacobian of LayerNorm

Differentiating $\hat{x}_i = (x_i-\mu)s^{-1}$:

$$\frac{\partial\hat{x}_i}{\partial x_j} = \frac{\partial(x_i-\mu)}{\partial x_j} \frac1s + (x_i-\mu) \frac{\partial s^{-1}}{\partial x_j}$$

Substituting the previously derived components:

$$\frac{\partial\hat{x}_i}{\partial x_j} = \frac{\delta_{ij}-1/D}{s} - \frac{(x_i-\mu)(x_j-\mu)}{Ds^3}$$

Since $(x_i-\mu)(x_j-\mu) = s^2\hat{x}_i\hat{x}_j$, this simplifies to the Jacobian matrix:

$$\frac{\partial\hat{x}_i}{\partial x_j} = \frac1s \left( \delta_{ij} - \frac1D - \frac{\hat{x}_i\hat{x}_j}{D} \right)$$

---

## 9. Multivariable Chain Rule

Since changing one input $x_j$ affects every normalized output $\hat{x}_i$:

$$\frac{\partial L}{\partial x_j} = \sum_i \frac{\partial L}{\partial\hat{x}_i} \frac{\partial\hat{x}_i}{\partial x_j} = \frac1s \sum_i d_i \left( \delta_{ij} - \frac1D - \frac{\hat{x}_i\hat{x}_j}{D} \right)$$

Breaking this down into three sum terms:

1. **First Term:** $\sum_i d_i\delta_{ij} = d_j$
2. **Second Term:** $\sum_i\frac{d_i}{D} = \frac1D\sum_i d_i$
3. **Third Term:** $\sum_i \frac{d_i\hat{x}_i\hat{x}_j}{D} = \frac{\hat{x}_j}{D} \sum_i d_i\hat{x}_i$

---

## 10. Final Vectorized Gradient

Combining all three terms, we arrive at the fundamental backward pass formula:

$$\frac{\partial L}{\partial x_j} = \frac1s \left[ d_j - \frac1D \sum_i d_i - \frac{\hat{x}_j}{D} \sum_i d_i\hat{x}_i \right]$$

---

## Summary

The vectorized expressions used directly in production frameworks (PyTorch, TensorFlow, JAX) are:

| Parameter / Variable | Gradient Formula |
| :--- | :--- |
| **Bias Gradient** ($\beta$) | $\frac{\partial L}{\partial\beta_i} = \frac{\partial L}{\partial y_i}$ |
| **Scale Gradient** ($\gamma$) | $\frac{\partial L}{\partial\gamma_i} = \frac{\partial L}{\partial y_i}\hat{x}_i$ |
| **Input Gradient** ($x$) | $\frac{\partial L}{\partial x_i} = \frac1{\sqrt{\sigma^2+\epsilon}} \left[ d_i - \operatorname{mean}(d) - \hat{x}_i \operatorname{mean}(d\odot\hat{x}) \right]$ |

*(where $d_i = \gamma_i \frac{\partial L}{\partial y_i}$ and $\odot$ denotes element-wise multiplication)*